# Pipeline de análise de defeitos de fabricação

Este notebook transforma o script `src/pipeline.py` em etapas interativas. Abra-o no VS Code com a extensão Jupyter e execute as células em ordem (`Shift+Enter`).

## 1. Carregamento dos dados

In [4]:
# %pip install pandas

In [5]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path.home() / 'Downloads' / 'manufacturing_defect_dataset.csv')

## 2. Inspeção inicial

In [6]:
# Exibe as primeiras linhas
print(df.head(20))

# Verifica valores nulos e linhas duplicadas
print(f'Valores nulos: \n{df.isnull().sum()}')
print(f'Total de linhas duplicadas: \n{df.duplicated().sum()}')

# Exibe o resumo estatístico
print(f'Resumo estatístico: \n{df.describe()}')

# Verifica valores negativos nas colunas de horas de manutenção,
# volume de produção e custo de produção
print(df[(df['MaintenanceHours'] < 0) |
         (df['ProductionVolume'] < 0) |
         (df['ProductionCost'] < 0)][
             ['MaintenanceHours', 'ProductionVolume', 'ProductionCost']
         ])

    ProductionVolume  ProductionCost  SupplierQuality  DeliveryDelay  \
0                202    13175.403783        86.648534              1   
1                535    19770.046093        86.310664              4   
2                960    19060.820997        82.132472              0   
3                370     5647.606037        87.335966              5   
4                206     7472.222236        81.989893              3   
5                171     6975.931602        95.331919              1   
6                800    15889.698650        99.325486              3   
7                120    17266.779948        99.401489              4   
8                714     8202.670495        97.301422              5   
9                221    12587.790394        92.015843              2   
10               566    17610.545420        94.017314              3   
11               314    15992.023171        85.704116              0   
12               430    13133.558082        92.615408           

## 3. Validação de regras de negócio e limites físicos

In [7]:
print('*** VALIDAÇÃO DE REGRAS DE NEGÓCIO ***')

# Verifica percentual de inatividade maior que 100%
downtime_invalido = df[df['DowntimePercentage'] > 100]
print(f'Quantidade de linhas com Downtime maior que 100%: {len(downtime_invalido)}')

# Verifica eficiência energética menor ou igual a zero
eficiencia_invalida = df[df['EnergyEfficiency'] <= 0]
print(f'Quantidade de linhas com Eficiência Energética menor ou igual a zero: {len(eficiencia_invalida)}')

*** VALIDAÇÃO DE REGRAS DE NEGÓCIO ***
Quantidade de linhas com Downtime maior que 100%: 0
Quantidade de linhas com Eficiência Energética menor ou igual a zero: 0


## 4. Detecção de outliers no custo unitário

Calcula o custo unitário e usa o intervalo interquartil (IQR) para identificar e filtrar outliers.

In [8]:
# Cria uma coluna para o custo unitário
df['UnitCost'] = df['ProductionCost'] / df['ProductionVolume']

# Calcula os quartis e o intervalo interquartil
q1 = df['UnitCost'].quantile(0.25)
q3 = df['UnitCost'].quantile(0.75)
iqr = q3 - q1

# Define os limites inferior e superior para outliers
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Filtra os outliers de custo unitário
outliers_custo = df[(df['UnitCost'] < limite_inferior) |
                    (df['UnitCost'] > limite_superior)]
print(f'Quantidade de lotes com anomalias de custo unitário: {len(outliers_custo)}')

# Mantém apenas os lotes dentro do limite saudável de custo
df_clean = df[(df['UnitCost'] >= limite_inferior) &
              (df['UnitCost'] <= limite_superior)].copy()

print(f'Total de linhas na base original: {len(df)}')
print(f'Total de linhas na base pós tratamento de custos: {len(df_clean)}')

Quantidade de lotes com anomalias de custo unitário: 284
Total de linhas na base original: 3240
Total de linhas na base pós tratamento de custos: 2956
